In [75]:
!pip install pandas numpy scikit-learn fairlearn matplotlib torch networkx requests pgmpy

In [76]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
from fairlearn.metrics import demographic_parity_difference, demographic_parity_ratio, selection_rate
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import networkx as nx
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from fairlearn.metrics import (
    demographic_parity_difference,
    demographic_parity_ratio,
    selection_rate,
)

In [77]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

In [78]:
#just creates a pipeline for preprocessing of standardscaler and one hot encoding
def get_preprocessor(numerical, categorical):
    numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
    category_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])
    preprocessor = ColumnTransformer(transformers=[
        ('num', numeric_transformer, numerical),
        ('cat', category_transformer, categorical)
    ])
    return preprocessor


In [79]:
#there is some error with the race thing. i cant get all the classes of it. else rest is normal analysis
def print_fairness_metrics(y_true, y_pred, sensitive_attr):
    # Ensure inputs are arrays
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    sensitive_attr = np.asarray(sensitive_attr)

    # Check that lengths match
    if not (len(y_true) == len(y_pred) == len(sensitive_attr)):
        raise ValueError(
            f"Inconsistent input lengths: y_true={len(y_true)}, y_pred={len(y_pred)}, sensitive_attr={len(sensitive_attr)}"
        )
    # Binarize probabilistic predictions
    if np.issubdtype(y_pred.dtype, np.floating):
        y_pred_binary = (y_pred > 0.5).astype(int)
    else:
        y_pred_binary = y_pred

    acc = accuracy_score(y_true, y_pred_binary)
    # Fairness metrics
    if len(np.unique(sensitive_attr)) < 2:
        print("⚠️ WARNING: Only one group in sensitive attribute. Fairness metrics may be misleading.")
        dpd = 0.0
        dpr = 1.0
    else:
        dpd = demographic_parity_difference(
            y_true=y_true, y_pred=y_pred_binary, sensitive_features=sensitive_attr
        )
        dpr = demographic_parity_ratio(
            y_true=y_true, y_pred=y_pred_binary, sensitive_features=sensitive_attr
        )
    # Display metrics
    print(f"\n📊 Fairness Metrics:")
    print(f"Accuracy: {acc:.4f}")
    print(f"Demographic Parity Difference: {dpd:.4f}")
    print(f"Demographic Parity Ratio: {dpr:.4f}")
    # Group-wise selection rates
    df_temp = pd.DataFrame({
        'true': y_true,
        'pred': y_pred_binary,
        'group': sensitive_attr
    })
    selection_rates = df_temp.groupby('group')['pred'].mean().sort_values()
    print(f"\n🎯 Selection Rates by Group:")
    print(selection_rates)
    # Per-group selection rates using fairlearn
    print("\n📈 Individual Group Selection Rates (Fairlearn):")
    for group in np.unique(sensitive_attr):
        group_mask = sensitive_attr == group
        group_sr = selection_rate(y_true[group_mask], y_pred_binary[group_mask])
        print(f"Group {group}: {group_sr:.4f}")


In [80]:
#a post processing technique.
def apply_statistical_reweighting(y_pred, sensitive_attr, target_rate=None):
    df = pd.DataFrame({'group': sensitive_attr, 'pred': y_pred})
    selection_rates = df.groupby('group')['pred'].mean() #find selection rate
    if len(selection_rates) < 2:
        print("WARNING: Only one group in sensitive attribute, reweighting has no effect")
        return y_pred
        
    if target_rate is None:
        target_rate = selection_rates.mean() #set it as target rate
    adjustment_map = {} #by how much does the target rate differ, how much to get there
    for group, rate in selection_rates.items():
        diff = target_rate - rate
        adjustment_map[group] = diff
    adjusted_preds = y_pred.copy() #create a copy of og for adjustment
    for group, adjustment in adjustment_map.items():
        idxs = np.where(df['group'] == group)[0] #indices of the group
        group_preds = y_pred[idxs] #predictions of the group
        #to convert float preds into binary if not already
        if isinstance(group_preds[0], (float, np.float32, np.float64)) and not isinstance(group_preds[0], bool):
            binary_preds = group_preds > 0.5 #binary serires of 0 and 1
        else:
            binary_preds = group_preds #as is if already binary
        #actual statistical reweighting, although it looks sus. have to check it out
        #basically, if the target rate is +ve, there are less +ves in the predictions. so i need to find the 0s and flip them into 1.
        #exact opp for -ve target rate
        if adjustment > 0: 
            flip_candidates = idxs[binary_preds == 0]
        else:
            flip_candidates = idxs[binary_preds == 1]
            
        if len(flip_candidates) > 0:
            #adjustment rate vs flip candidates (min cause this sounds so sus)
            num_to_flip = min(int(abs(adjustment) * len(idxs)), len(flip_candidates))
            flip_idxs = np.random.choice(flip_candidates, size=num_to_flip, replace=False)
            #replace all indices, randomly, only once with num_to_flip(0 or 1 acc to target rate)
            #flip acc to type of data
            if isinstance(y_pred[0], (float, np.float32, np.float64)) and not isinstance(y_pred[0], bool):
                adjusted_preds[flip_idxs] = 1.0 if adjustment > 0 else 0.0
            else:
                adjusted_preds[flip_idxs] = 1 if adjustment > 0 else 0
    
    return adjusted_preds

In [81]:
#convert adjacency matrix into a diGraph and check if its a DAG for the BBN
def is_dag(adj_matrix):
    G = nx.DiGraph(adj_matrix)
    return nx.is_directed_acyclic_graph(G)
    
#uses dirichelt's dist to generate 1 conditional prob table for each node, where each table has 2 vals (for 1 and 0)
#dirichelt's dist => probabilites over a dist. for eg - coin toss once = [0.4,0.6]
#sums upto 1, positive and reps probabilties
def random_cpt(num_nodes):
    return [np.random.dirichlet(np.ones(2), size=1)[0] for _ in range(num_nodes)]

def generate_random_bbn(num_nodes):
    #this generates a lower triangular matrix. apparently this makes a DAG and prevents cycles from forming
    #generate a nXn lower triangular boolean matrix with threshold 0.7
    adj = np.tril((np.random.rand(num_nodes, num_nodes) > 0.7).astype(int), -1)
    #check if its a DAG for BBN else do it again
    if not is_dag(adj):
        return generate_random_bbn(num_nodes)
    #generate conditional probability table
    cpts = random_cpt(num_nodes)
    return (adj, cpts)

def bbn_fitness(model_preds, sensitive, bbn_chromosome):
    adj, cpts = bbn_chromosome #unpack a chromosome
    node_preds = []
    for i, (w0, w1) in enumerate(cpts):
        combined = model_preds * w1 + (1 - model_preds) * w0
        node_preds.append(combined) #soft probabilistic op of each node of BBN to eval the fitness
    final_pred = (np.mean(np.stack(node_preds), axis=0) > 0.5).astype(int) #consider final pred with mean
    # Check if there's only one group
    if len(np.unique(sensitive)) < 2:
        # Return a relatively low fitness so the algorithm knows to improve
        return 0.5
    #use DP as fitness score (lower dp, higher fitness hence the 1- )
    fairness_score = 1.0 - abs(demographic_parity_difference(y_true=sensitive, y_pred=final_pred, sensitive_features=sensitive))
    return fairness_score

def crossover(p1, p2):
    #unpack each parent chromosome
    adj1, cpt1 = p1
    adj2, cpt2 = p2
    #mix the two matrices to create a new matrix in the shape of adj1. 
    #whichever condn is true, take that particular val (mat(1X2) IS 0.2 AND 0.6 take 0.6)
    child_adj = np.where(np.random.rand(*adj1.shape) > 0.5, adj1, adj2)
    #resulting structure should be a DAG, else revert to prev structure
    if not is_dag(child_adj):
        child_adj = adj1
    #similarly with the conditional prob table as well
    child_cpt = [(c1 if random.random() > 0.5 else c2) for c1, c2 in zip(cpt1, cpt2)]
    return (child_adj, child_cpt)

def mutate(chromosome):
    adj, cpts = chromosome
    num_nodes = len(cpts)
    new_adj = adj.copy()
    #generate to random nums to mutate
    i, j = random.randint(0, num_nodes-1), random.randint(0, num_nodes-1)
    #flip that position (edge)
    if i != j and i > j:
        new_adj[i, j] = 1 - new_adj[i, j]
    #result must be a DAG
    if not is_dag(new_adj):
        new_adj = adj
    #mutates the CPT by adding some noise and modifying the vals
    new_cpts = []
    for c in cpts:
        noise = np.random.normal(0, 0.05, size=2)
        new_c = np.clip(c + noise, 0.01, 0.99)
        new_c /= new_c.sum()
        new_cpts.append(new_c)
    return (new_adj, new_cpts)


In [82]:
def evolve_bbn_adversary(model_preds, sensitive, num_nodes=5, pop_size=10, generations=50):
    population = [generate_random_bbn(num_nodes) for _ in range(pop_size)] #generate your population
    #calc fitness for all
    best_fitness = 0
    for gen in range(generations):
        fitness_scores = [bbn_fitness(model_preds, sensitive, chrom) for chrom in population]
        # Track best fitness
        gen_best_fitness = max(fitness_scores) #best fit of generation
        best_fitness = max(best_fitness, gen_best_fitness) #global best fitness
        #top 2 fittest BBNs are now parents
        top_idx = np.argsort(fitness_scores)[-2:]
        elites = [population[i] for i in top_idx]
        #new population building
        new_pop = elites.copy()
        while len(new_pop) < pop_size:
            p1, p2 = random.sample(elites, 2)
            child = crossover(p1, p2)
            if random.random() < 0.3:
                child = mutate(child)
            new_pop.append(child)
        #replace population
        population = new_pop
        print(f"[Gen {gen}] Best Fairness: {gen_best_fitness:.4f}")
    #return fittest chromosome or BBN with the highest fitness after everything    
    best_idx = np.argmax([bbn_fitness(model_preds, sensitive, chrom) for chrom in population])
    return population[best_idx]

def run_evolved_bbn_adversary(model_preds, sensitive_attr, num_nodes=5, pop_size=10, generations=50):
    print("\n[Running Genetic Algorithm to evolve BBN adversary...]")
    # Check if there's only one group
    if len(np.unique(sensitive_attr)) < 2:
        print("WARNING: Only one group in sensitive attribute, BBN adversary will not be effective")
    #find the best BBN
    best_chromosome = evolve_bbn_adversary(
        model_preds=model_preds,
        sensitive=sensitive_attr,
        num_nodes=num_nodes,
        pop_size=pop_size,
        generations=generations
    )
    #unpack it
    adj, cpts = best_chromosome
    adjusted_preds = []
    #see the prediction of the model for adjusted
    for i, (w0, w1) in enumerate(cpts):
        combined = model_preds * w1 + (1 - model_preds) * w0
        adjusted_preds.append(combined)
    final_pred = (np.mean(np.stack(adjusted_preds), axis=0) > 0.5).astype(int)
    # Calculate metrics against original predictions
    binary_model_preds = (model_preds > 0.5).astype(int)
    accuracy = accuracy_score(binary_model_preds, final_pred)
    # Check if there's only one group
    if len(np.unique(sensitive_attr)) < 2:
        fairness = 0.5  # Placeholder value
    else:
        fairness = 1.0 - abs(demographic_parity_difference(
            y_true=sensitive_attr, 
            y_pred=final_pred, 
            sensitive_features=sensitive_attr
        ))
    
    print(f"\n[Evolved BBN Adversary Results]")
    print(f"Prediction Preservation: {accuracy:.4f}")
    print(f"Fairness (1 - |DPD|): {fairness:.4f}")
    
    return best_chromosome, final_pred

In [83]:
class AdversarialFairClassifier:
    def __init__(self, input_dim, hidden_dim=64, epochs=20, batch_size=128, lr=0.001, lambda_param=0.8,
                 bbn_nodes=5, bbn_pop_size=10, bbn_generations=50): #might have to tune these hyperparams
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.lambda_param = lambda_param
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        self.predictor = nn.Sequential(#a simple predictor with ReLU activation (Multi Layer Perceptron)
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(), #introduce non linearity as relu is a piecewise func
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid() #squishification with sigmoid
        ).to(self.device)
        #adam optimizer. minimizes the loss by updating the wts using step function
        self.optimizer = torch.optim.Adam(self.predictor.parameters(), lr=lr)
        # BBN Adversary Parameters
        self.bbn_nodes = bbn_nodes
        self.bbn_pop_size = bbn_pop_size
        self.bbn_generations = bbn_generations
        self.best_bbn = None
        
    def fit(self, X, y, sensitive_attr):
        if hasattr(X, "toarray"):
            X = X.toarray()  
        if len(np.unique(sensitive_attr)) < 2: #warning. have to fix it
            print("WARNING: Only one group in sensitive attribute, BBN adversary will not be effective")
        #convert to a tensor
        X_tensor = torch.FloatTensor(X).to(self.device) 
        y_tensor = torch.FloatTensor(y).view(-1, 1).to(self.device)
        dataset = TensorDataset(X_tensor, y_tensor)
        dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True) #loads data in batches
        for epoch in range(self.epochs):
            total_loss = 0
            self.predictor.train() #training mode
            for X_batch, y_batch in dataloader:
                self.optimizer.zero_grad() #reset gradients for optimizer
                y_pred = self.predictor(X_batch)#predict
                loss = nn.BCELoss()(y_pred, y_batch) #compute Binary Cross Entropy
                loss.backward() #backpropogate
                self.optimizer.step() #update wts
                total_loss += loss.item() #compute loss
            if epoch % 5 == 0:
                print(f"Epoch {epoch}: Predictor Loss: {total_loss / len(dataloader):.4f}")
        
        with torch.no_grad():
            model_preds = self.predictor(torch.FloatTensor(X).to(self.device)).cpu().numpy().flatten()
        self.best_bbn, self.final_preds = self._run_bbn_adversary(model_preds, sensitive_attr) 
        #better the preds with BBN adversary

    #below gives better predictions with the best BBN
    def _run_bbn_adversary(self, model_preds, sensitive_attr):
        print("\n[Running Genetic Algorithm to evolve BBN adversary...]")
        if len(np.unique(sensitive_attr)) < 2:
            print("WARNING: Only one group in sensitive attribute, BBN adversary will not be effective")
        # Evolve the BBN structure for DEMOGRAPHIC PARITY
        best_chromosome = evolve_bbn_adversary(
            model_preds=model_preds,
            sensitive=sensitive_attr,
            num_nodes=self.bbn_nodes,
            pop_size=self.bbn_pop_size,
            generations=self.bbn_generations
        )
        
        adj, cpts = best_chromosome
        adjusted_preds = []
        for i, (w0, w1) in enumerate(cpts):
            combined = model_preds * w1 + (1 - model_preds) * w0
            adjusted_preds.append(combined)
        final_pred = (np.mean(np.stack(adjusted_preds), axis=0) > 0.5).astype(int)
        
        accuracy = accuracy_score((model_preds > 0.5).astype(int), final_pred)
        fairness = 1.0 - abs(demographic_parity_difference(
            y_true=sensitive_attr, 
            y_pred=final_pred, 
            sensitive_features=sensitive_attr
        )) if len(np.unique(sensitive_attr)) >= 2 else 0.5

        print(f"\n[Evolved BBN Adversary Results]")
        print(f"Prediction Preservation: {accuracy:.4f}")
        print(f"Fairness (1 - |DPD|): {fairness:.4f}")
        return best_chromosome, final_pred

    
    def predict(self, X):
        if hasattr(X, "toarray"):
            X = X.toarray()
        X_tensor = torch.FloatTensor(X).to(self.device)
        # Get predictions for the new data instead of using stored predictions
        with torch.no_grad():
            preds = self.predictor(X_tensor).cpu().numpy().flatten()
        # Apply the BBN transformation to the new predictions
        if self.best_bbn:
            adj, cpts = self.best_bbn
            adjusted_preds = []
            for i, (w0, w1) in enumerate(cpts):
                combined = preds * w1 + (1 - preds) * w0
                adjusted_preds.append(combined)
            
            final_pred = (np.mean(np.stack(adjusted_preds), axis=0) > 0.5).astype(int)
            return final_pred
        # If no BBN has been trained, just return binary predictions
        return (preds > 0.5).astype(int)
    
    def predict_proba(self, X):
        if hasattr(X, "toarray"):
            X = X.toarray()
        X_tensor = torch.FloatTensor(X).to(self.device)
        with torch.no_grad():
            return self.predictor(X_tensor).cpu().numpy().flatten()


In [84]:
def hybrid_fairness_mitigation(model_preds, sensitive_attr, y_true, target_rate=None, 
                              num_nodes=5, pop_size=10, generations=20, 
                              weight_reweighting=0.5):
    import numpy as np
    import pandas as pd
    from sklearn.metrics import accuracy_score
    from fairlearn.metrics import demographic_parity_difference, demographic_parity_ratio

    print("\n[Running Hybrid Fairness Mitigation...]")

    if len(np.unique(sensitive_attr)) < 2:
        print("WARNING: Only one group in sensitive attribute.")
        return model_preds, {
            'accuracy': 1.0,
            'fairness': 0.5,
            'dpd': 0.0,
            'dpr': 1.0,
            'selection_rates': pd.Series({'0': np.mean((model_preds > 0.5).astype(int))})
        }

    # Run both mitigation methods
    reweighted_preds = apply_statistical_reweighting(model_preds.copy(), sensitive_attr, target_rate)
    _, bbn_preds = run_evolved_bbn_adversary(
        model_preds=model_preds, 
        sensitive_attr=sensitive_attr,
        num_nodes=num_nodes,
        pop_size=pop_size,
        generations=generations
    )

    # Convert predictions to binary or keep probabilities
    def to_prob_and_bin(preds):
        if isinstance(preds[0], (float, np.floating)) and not isinstance(preds[0], bool):
            return preds, (preds > 0.5).astype(int)
        return preds.astype(float), preds

    reweighted_probs, reweighted_bin = to_prob_and_bin(reweighted_preds)
    bbn_probs, bbn_bin = to_prob_and_bin(bbn_preds)
    
    #agree mask
    agree_mask = (reweighted_bin == bbn_bin)
    final_preds = np.where(agree_mask, 
                           reweighted_bin, 
                           (weight_reweighting * reweighted_probs + (1 - weight_reweighting) * bbn_probs > 0.5).astype(int))

    # Calculate metrics
    accuracy = accuracy_score(y_true, final_preds)
    
    # Include y_true in the fairlearn metric calls
    dpd = demographic_parity_difference(
        y_true=y_true,
        y_pred=final_preds, 
        sensitive_features=sensitive_attr
    )
    
    dpr = demographic_parity_ratio(
        y_true=y_true,
        y_pred=final_preds, 
        sensitive_features=sensitive_attr
    )
    
    fairness = 1.0 - abs(dpd)

    selection_rates = pd.DataFrame({'pred': final_preds, 'group': sensitive_attr}).groupby('group')['pred'].mean()

    print(f"\n[Hybrid Results] Accuracy: {accuracy:.4f}, Fairness: {fairness:.4f}, DPD: {dpd:.4f}, DPR: {dpr:.4f}")
    print("Selection Rates by Group:\n", selection_rates)

    return final_preds, {
        'accuracy': accuracy,
        'fairness': fairness,
        'dpd': dpd,
        'dpr': dpr,
        'selection_rates': selection_rates
    }

In [85]:
def sequential_hybrid_fairness_mitigation(model_preds, sensitive_attr, y_true, 
                                         num_nodes=5, pop_size=10, generations=20):
    """
    Apply statistical reweighting first, then use the reweighted predictions
    as input to the BBN adversary for further fairness improvement.
    """
    print("\n[Running Sequential Hybrid Fairness Mitigation...]")

    if len(np.unique(sensitive_attr)) < 2:
        print("WARNING: Only one group in sensitive attribute.")
        return model_preds, {
            'accuracy': 1.0,
            'fairness': 0.5,
            'dpd': 0.0,
            'dpr': 1.0,
            'selection_rates': pd.Series({'0': np.mean((model_preds > 0.5).astype(int))})
        }

    # Step 1: Apply statistical reweighting
    reweighted_preds = apply_statistical_reweighting(model_preds.copy(), sensitive_attr)
    
    # Step 2: Use reweighted predictions as input to BBN adversary
    best_bbn, bbn_adjusted_preds = run_evolved_bbn_adversary(
        model_preds=reweighted_preds,  # Use reweighted predictions as input
        sensitive_attr=sensitive_attr,
        num_nodes=num_nodes,
        pop_size=pop_size,
        generations=generations
    )
    
    # Calculate metrics for the final predictions
    accuracy = accuracy_score(y_true, bbn_adjusted_preds)
    
    dpd = demographic_parity_difference(
        y_true=y_true,
        y_pred=bbn_adjusted_preds, 
        sensitive_features=sensitive_attr
    )
    
    dpr = demographic_parity_ratio(
        y_true=y_true,
        y_pred=bbn_adjusted_preds, 
        sensitive_features=sensitive_attr
    )
    
    fairness = 1.0 - abs(dpd)

    selection_rates = pd.DataFrame({'pred': bbn_adjusted_preds, 
                                   'group': sensitive_attr}).groupby('group')['pred'].mean()

    print(f"\n[Sequential Hybrid Results] Accuracy: {accuracy:.4f}, Fairness: {fairness:.4f}, DPD: {dpd:.4f}, DPR: {dpr:.4f}")
    print("Selection Rates by Group:\n", selection_rates)

    return bbn_adjusted_preds, {
        'accuracy': accuracy,
        'fairness': fairness,
        'dpd': dpd,
        'dpr': dpr,
        'selection_rates': selection_rates
    }

In [86]:
def reverse_sequential_hybrid(model_preds, sensitive_attr, y_true, 
                             num_nodes=5, pop_size=10, generations=20):
    """
    Apply BBN adversary first, then use the adjusted predictions
    as input to statistical reweighting for further fairness improvement.
    """
    # Step 1: Apply BBN adversary
    _, bbn_adjusted_preds = run_evolved_bbn_adversary(
        model_preds=model_preds,
        sensitive_attr=sensitive_attr,
        num_nodes=num_nodes,
        pop_size=pop_size,
        generations=generations
    )
    
    # Step 2: Apply statistical reweighting to BBN adjusted predictions
    final_preds = apply_statistical_reweighting(bbn_adjusted_preds, sensitive_attr)
    
     # Calculate metrics for the final predictions
    accuracy = accuracy_score(y_true, final_preds)
    
    dpd = demographic_parity_difference(
        y_true=y_true,
        y_pred=final_preds, 
        sensitive_features=sensitive_attr
    )
    
    dpr = demographic_parity_ratio(
        y_true=y_true,
        y_pred=final_preds, 
        sensitive_features=sensitive_attr
    )
    
    fairness = 1.0 - abs(dpd)

    selection_rates = pd.DataFrame({'pred': final_preds, 
                                   'group': sensitive_attr}).groupby('group')['pred'].mean()

    print(f"\n[Sequential Hybrid Results] Accuracy: {accuracy:.4f}, Fairness: {fairness:.4f}, DPD: {dpd:.4f}, DPR: {dpr:.4f}")
    print("Selection Rates by Group:\n", selection_rates)

    return bbn_adjusted_preds, {
        'accuracy': accuracy,
        'fairness': fairness,
        'dpd': dpd,
        'dpr': dpr,
        'selection_rates': selection_rates
    }

In [87]:
def adaptive_sequential_hybrid(model_preds, sensitive_attr, y_true, 
                              num_nodes=5, pop_size=10, generations=20,
                              weight_reweighting=0.7):  # How much to prioritize fairness vs accuracy
    # Try both orders
    forward_preds, forward_metrics = sequential_hybrid_fairness_mitigation(
        model_preds, sensitive_attr, y_true, num_nodes, pop_size, generations)
    
    reverse_preds, reverse_metrics = reverse_sequential_hybrid(
        model_preds, sensitive_attr, y_true, num_nodes, pop_size, generations)
    
    # Calculate combined scores (weighted average of accuracy and fairness)
    forward_score = (1 - weight_reweighting) * forward_metrics['accuracy'] + weight_reweighting * forward_metrics['fairness']
    reverse_score = (1 - weight_reweighting) * reverse_metrics['accuracy'] + weight_reweighting * reverse_metrics['fairness']
    
    # Return the best approach
    if forward_score >= reverse_score:
        print("Forward sequential hybrid performed better")
        return forward_preds, forward_metrics
    else:
        print("Reverse sequential hybrid performed better")
        return reverse_preds, reverse_metrics

In [88]:
def fairness_comparison(X_test, y_test, base_preds, sensitive_attr, 
                        use_bbn_adversary=True, adv_model=None, adv_data=None,
                        use_hybrid=True, hybrid_weight=0.5):
    print("=" * 40)
    print("[1] Base Model")
    print("=" * 40)
    #basic normal model
    print_fairness_metrics(y_test, base_preds, sensitive_attr)
    
    print("\n" + "=" * 40)
    print("[2] Postprocessed (Statistical Reweighting)")
    print("=" * 40)
    reweighted_preds = apply_statistical_reweighting(base_preds.copy(), sensitive_attr)
    #postprocessing
    print_fairness_metrics(y_test, reweighted_preds, sensitive_attr)
    
    if use_bbn_adversary:
        print("\n" + "=" * 40)
        print("[3] Adversarial BBN-Debiased Model")
        print("=" * 40)
        _, bbn_adjusted_preds = run_evolved_bbn_adversary(model_preds=base_preds, sensitive_attr=sensitive_attr)
        print_fairness_metrics(y_test, bbn_adjusted_preds, sensitive_attr)
    
    if adv_model is not None and adv_data is not None:
        print("\n" + "=" * 40)
        print("[4] External Adversarial Model")
        print("=" * 40)
        try:
            adv_preds = adv_model.predict(adv_data)
            # Check that lengths match
            if len(y_test) == len(adv_preds) == len(sensitive_attr):
                print_fairness_metrics(y_test, adv_preds, sensitive_attr)
            else:
                print(f"Error: Inconsistent lengths - y_test: {len(y_test)}, adv_preds: {len(adv_preds)}, sensitive_attr: {len(sensitive_attr)}")
                print("Skipping fairness metrics for adversarial model.")
        except Exception as e:
            print(f"Error predicting with adversarial model: {e}")
            print("Skipping fairness metrics for adversarial model.")
    
    #hybrid - pass y_test as required
    if use_hybrid:
        print("\n" + "=" * 40)
        print("[5] Hybrid (Reweighting + BBN) Approach")
        print("=" * 40)
        hybrid_preds, _ = hybrid_fairness_mitigation(
            model_preds=base_preds,
            sensitive_attr=sensitive_attr,
            y_true=y_test,  # Make sure to pass y_test
            weight_reweighting=hybrid_weight
        )
        print_fairness_metrics(y_test, hybrid_preds, sensitive_attr)
    if use_hybrid:
        print("\n" + "=" * 40)
        print("[6] Adaptive Hybrid (Reweighting + BBN) Approach")
        print("=" * 40)
        hybrid_preds1, _ = adaptive_sequential_hybrid(
            model_preds=base_preds,
            sensitive_attr=sensitive_attr,
            y_true=y_test,  # Make sure to pass y_test
            weight_reweighting=hybrid_weight
        )
        print_fairness_metrics(y_test, hybrid_preds1, sensitive_attr)
        
    
    print("\nNote: Lower Demographic Parity Difference and Ratio closer to 1.0 indicate better fairness.")

In [89]:

def main():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
    column_names = [
        "age", "workclass", "fnlwgt", "education", "education-num", 
        "marital-status", "occupation", "relationship", "race", "sex", 
        "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
    ]
    print("LOADING REAL UCI")
    data = pd.read_csv(url, names=column_names, sep=",\s*", engine="python")  # ✅ fix here
    print(f"Dataset shape: {data.shape}")
    print(f"Columns: {data.columns.tolist()}")
    # Check target distribution before encoding
    print("\nTarget distribution:")
    print(data["income"].value_counts()) 
    # Convert income to 0 and 1
    data["income"] = np.where(data["income"] == ">50K", 1, 0)
    # Print values and new distribution
    print("\nEncoded target values:")
    print(data["income"].values)
    print("\nNew target distribution:")
    print(data["income"].value_counts())
    numerical_cols = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
    categorical_cols = ["workclass", "education", "marital-status", "occupation", "relationship", "native-country"]
    X = data.drop(["income", "sex", "race"], axis=1)
    y = data["income"].values
    # Race mapping for sensitive attribute
    race_mapping = {
        "White": 0,
        "Black": 1,
        "Asian-Pac-Islander": 2,
        "Amer-Indian-Eskimo": 3,
        "Other": 4
    }
    # Safe and clean mapping
    sensitive_attribute = data["race"].map(race_mapping)
    print(f"\nSensitive attribute categories: {np.unique(sensitive_attribute)}")
    print(f"Category counts: {pd.Series(sensitive_attribute).value_counts().to_dict()}")
    X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
        X, y, sensitive_attribute, test_size=0.2, random_state=RANDOM_SEED, stratify=y
    )
    
    print("\nPreprocessing data...")
    preprocessor = get_preprocessor(numerical_cols, categorical_cols)
    X_train_proc = preprocessor.fit_transform(X_train)
    X_test_proc = preprocessor.transform(X_test)
    print("\nTraining base logistic regression model...")
    base_model = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
    base_model.fit(X_train_proc, y_train)
    
    base_preds = base_model.predict(X_test_proc)
    base_probs = base_model.predict_proba(X_test_proc)[:, 1]
    
    print("\nTraining adversarial fair classifier...")
    input_dim = X_train_proc.shape[1]
    adv_model = AdversarialFairClassifier(input_dim=input_dim, epochs=10)
    adv_model.fit(X_train_proc, y_train, s_train)
    
    adv_preds = adv_model.predict(X_test_proc)
    
    print("\nApplying hybrid fairness mitigation...")
    # In the main function:
    hybrid_preds, hybrid_metrics = hybrid_fairness_mitigation(
        model_preds=base_probs,
        sensitive_attr=s_test,
        y_true=y_test,  # Pass y_test as required
        weight_reweighting=0.6
    )

    print("\nComparing fairness metrics across models...")
    # In the main function:
    fairness_comparison(
        X_test=X_test_proc,
        y_test=y_test,
        base_preds=base_probs,
        sensitive_attr=s_test,
        adv_model=adv_model,
        adv_data=X_test_proc,
        hybrid_weight=0.6
    )
    
    print("\nDone!")

if __name__ == "__main__":
    main()

LOADING REAL UCI
Dataset shape: (32561, 15)
Columns: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

Target distribution:
income
<=50K    24720
>50K      7841
Name: count, dtype: int64

Encoded target values:
[0 0 0 ... 0 0 1]

New target distribution:
income
0    24720
1     7841
Name: count, dtype: int64

Sensitive attribute categories: [0 1 2 3 4]
Category counts: {0: 27816, 1: 3124, 2: 1039, 3: 311, 4: 271}

Preprocessing data...

Training base logistic regression model...

Training adversarial fair classifier...
Epoch 0: Predictor Loss: 0.3890
Epoch 5: Predictor Loss: 0.3020

[Running Genetic Algorithm to evolve BBN adversary...]
[Gen 0] Best Fairness: 0.8374
[Gen 1] Best Fairness: 0.8374
[Gen 2] Best Fairness: 0.8374
[Gen 3] Best Fairness: 0.8374
[Gen 4] Best Fairness: 0.8374
[Gen 5] Best Fairness: 0.8374
[Gen 6] Best Fairness:

In [90]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
column_names = [
    "age", "workclass", "fnlwgt", "education", "education-num", 
    "marital-status", "occupation", "relationship", "race", "sex", 
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]
print("LOADING REAL UCI")
data = pd.read_csv(url, names=column_names, sep=",\s*", engine="python")  # ✅ fix here
print(f"Dataset shape: {data.shape}")
print(f"Columns: {data.columns.tolist()}")
# Check target distribution before encoding
print("\nTarget distribution:")
print(data["income"].value_counts()) 
# Convert income to 0 and 1
data["income"] = np.where(data["income"] == ">50K", 1, 0)
# Print values and new distribution
print("\nEncoded target values:")
print(data["income"].values)
print("\nNew target distribution:")
print(data["income"].value_counts())
print(data["race"].value_counts())
print(data["race"].unique())

LOADING REAL UCI
Dataset shape: (32561, 15)
Columns: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

Target distribution:
income
<=50K    24720
>50K      7841
Name: count, dtype: int64

Encoded target values:
[0 0 0 ... 0 0 1]

New target distribution:
income
0    24720
1     7841
Name: count, dtype: int64
race
White                 27816
Black                  3124
Asian-Pac-Islander     1039
Amer-Indian-Eskimo      311
Other                   271
Name: count, dtype: int64
['White' 'Black' 'Asian-Pac-Islander' 'Amer-Indian-Eskimo' 'Other']


Problems i have given up on - 
1. the redundancy of it all. its repeating right. we need to optimize it. the bbn adversary is called everytime. how abt call it once and use that everywhere



Future Goals.
1. i think we should implement everything for 1 dataset and then just duplicate that for the rest
2. this is just for demographic parity, we can use all other metrics
3. proper preprocessing and visuals and comparisons